# Kinthread: reworded questions with an open model (free Colab T4)

Use a Google account **other than the training one** (this uses its free GPU time).
1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**; allow Google Drive when asked.
2. Keep the tab open. The output grows on Drive in chunks; if the session ends, Run all again and it resumes.

An open model (Qwen3-4B-Instruct-2507, Apache-2.0) rewords engine-verified questions in other exam styles; a separate
reading of each new text must give the same facts and answer (`kinlogic/reword_local.py`, the checks of `kinlogic/reword.py`).
Only rewordings that pass are kept. The locked test set is never used.

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
REPO = '/content/Kinthread-llm-model'
if not os.path.exists(REPO):
    !git clone --depth 1 -b v9-reader https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), 'Google Drive did not mount: click Allow in the pop-up and run this cell again'
OUT = '/content/drive/MyDrive/kinthread/reworded_local.jsonl.gz'
os.makedirs(os.path.dirname(OUT), exist_ok=True)
print('rewordings ->', OUT)

## Install vLLM (about 3-5 minutes; falls back to transformers if it cannot run on this GPU)

In [ ]:
!pip -q install vllm 2>&1 | tail -2
!python -c "import vllm; print('vllm', vllm.__version__)" || echo 'vLLM not importable: transformers will be used' 

## Reword (thousands per hour; resumable)

In [ ]:
!python -m kinlogic.reword_local --src data/dataset_v2 --out {OUT} --n 20000

## Download the result (or copy it from Drive: MyDrive/kinthread/reworded_local.jsonl.gz)

In [ ]:
import gzip, json, shutil
n = sum(1 for _ in gzip.open(OUT, 'rt'))
print(n, 'verified rewordings')
shutil.copy(OUT, '/content/reworded_local.jsonl.gz')
from google.colab import files
files.download('/content/reworded_local.jsonl.gz')